# 07 · Final comparison — §3, §4, §6

Builds every comparison table in the report from committed results. In `synthetic` / `debug` mode it previews whatever has landed without writing anything.

| | |
|---|---|
| **Owner** | Member 4 |
| **Reads** | committed JSON only: `results/metrics/**` |
| **Writes (official)** | `results/tables/*.md`, trade-off figures, `report/figures/` |
| **Run it** | after notebook 06 — no GPU needed, runs in seconds |

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# On a Colab runtime (in the browser, or from VS Code via the Colab extension) it
# clones REPO at BRANCH, installs it, and on every re-run pulls what you pushed since.
# Locally the package is already installed, so this is skipped. Guide: COLAB.md
import importlib.util
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess

    def _secret(name, default):  # your Colab Secret if you set one, else the default
        try:
            from google.colab import userdata

            return userdata.get(name) or default
        except Exception:  # not set, or unreadable here (VS Code cannot read Secrets)
            return default

    # The code to run is REPO @ BRANCH. Set your own fork / branch once, in Colab Secrets,
    # as A03_REPO / A03_BRANCH. In VS Code, edit the defaults below for the session and
    # put them back before committing. See COLAB.md.
    REPO = _secret("A03_REPO", "ThejithaR/EN3150-Assignment-03-CNN")
    BRANCH = _secret("A03_BRANCH", "main")
    REPO_DIR = "/content/en3150-a03"

    def _git(*args):  # never prompts: a failure shows as an error, not a hang
        subprocess.run(["git", *args], check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

    url = f"https://github.com/{REPO}.git"
    if not os.path.exists(REPO_DIR):
        _git("clone", "--quiet", "--branch", BRANCH, url, REPO_DIR)
    else:  # re-running this cell picks up everything pushed since
        _git("-C", REPO_DIR, "remote", "set-url", "origin", url)
        _git("-C", REPO_DIR, "fetch", "--quiet", "--prune", "origin")
        _git("-C", REPO_DIR, "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    _git("log", "--oneline", "-1")  # prints the exact commit this run uses

In [ ]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths
from IPython.display import Markdown

from edgecnn.evaluation.reporting import (
    build_custom_comparison_table,
    build_final_comparison_table,
    build_optimizer_comparison_table,
    check_same_hardware,
    export_report_figures,
    load_all_runs,
    plot_tradeoff_scatter,
)

In [ ]:
# Shared figure style (Member 1): small inline figures, print-quality saved PNGs.
from edgecnn.evaluation.plotting import apply_style

apply_style()

## Write switch

Tables and figures span several runs, so this notebook uses one switch.

In [ ]:
WRITE = MODE == "official"

## 1 · What has landed

In [ ]:
runs = load_all_runs()
sorted(runs)

## 2 · Same hardware within each table?

Epoch times are only comparable within a table if those runs trained on the same runtime.

In [ ]:
tables = load_config("configs/experiments/model_b__adam.yaml").section("reporting")["tables"]
{table["name"]: check_same_hardware(table["runs"]) for table in tables}

## 3 · §3 — optimizer comparison

In [ ]:
Markdown(build_optimizer_comparison_table(write=WRITE))

## 4 · §4 — Model A vs Model B

In [ ]:
Markdown(build_custom_comparison_table(write=WRITE))

## 5 · §6 — Model B vs the SOTA backbones

In [ ]:
Markdown(build_final_comparison_table(write=WRITE))

## 6 · The trade-off in one picture

Parameters track memory; MACs track compute. A model can be cheap in one and expensive in the other.

In [ ]:
plot_tradeoff_scatter("trainable_params", write=WRITE)

In [ ]:
plot_tradeoff_scatter("macs", write=WRITE)

## 7 · Export figures for the report

In [ ]:
export_report_figures(write=WRITE)

## Discussion prompts — §6

- State ratios, not just both numbers: "N× fewer parameters for M points of accuracy".
- The 64×64 handicap on the SOTA backbones — say so, rather than claiming a clean win.
- Pretrained features matter most with scarce data; would the answer flip at 2,700 images?
- Peak activation memory, not parameter count, is often what blocks MCU deployment.
- "It depends on the memory budget" is strong only if you name the threshold where the answer flips.

## Official-run checklist

- [ ] `MODE = "official"` in the setup cell
- [ ] Kernel → **Restart & Run All** finished without errors
- [ ] every output is what you expect — no stray warnings or giant prints
- [ ] no hardware warnings above, or they are explained in the report
- [ ] `pytest tests/contracts` is green
- [ ] commit this notebook **with its outputs**, together with the files it wrote

In [ ]:
# ── Colab only: send this run's results back to GitHub ─────────────────────
# Does nothing locally, or in synthetic / debug mode. Guide: COLAB.md
if IN_COLAB and MODE == "official":
    from edgecnn.utils import push_results

    push_results(
        [paths.TABLES_DIR, paths.FIGURES_DIR, paths.REPORT_FIGURES_DIR],
        "Add comparison tables and figures (Colab)",
    )